In [ ]:
# 05_select_stations_for_inversion.py
import pandas as pd
df = pd.read_csv("determinant_station_summary_with_slope.csv")
meta = pd.read_csv("each_file_skew_classification.csv")
df = df.merge(meta[["Site","Is1D"]], on="Site", how="left")

# selection criteria (adjust thresholds as needed)
sel = df[
    (df["Is1D"]==True) &
    (df["Nperiods"]>=10) &
    (df["Slope_mid_1_100s"].abs() <= 0.15)
].copy()

sel = sel.sort_values(["Is1D","Nperiods"], ascending=[False, False])
sel[["Site","Lat","Lon","Nperiods","Slope_mid_1_100s","rho_T10s"]].to_csv("stations_to_invert.csv", index=False)
print("Wrote stations_to_invert.csv with", len(sel), "stations")


In [ ]:
# 06_prepare_inversion_inputs.py
import os, pandas as pd, glob
sel = pd.read_csv("stations_to_invert.csv")
per_dir = "determinant_per_station"
outdir = "inversion_inputs"
os.makedirs(outdir, exist_ok=True)

def site_to_fname(site):
    return site.replace(" ", "_").replace(",","")

for site in sel["Site"]:
    fname = os.path.join(per_dir, site_to_fname(site) + "_det.csv")
    if not os.path.exists(fname):
        print("Missing det CSV for", site); continue
    df = pd.read_csv(fname)
    # simple error estimate: 10% of rho
    df["rho_err"] = 0.1 * df["rho_det"].abs()
    df[["period_s","Zdet_real","Zdet_imag","rho_det","rho_err"]].to_csv(os.path.join(outdir, site_to_fname(site) + "_inv_input.csv"), index=False)
    print("Prepared", site)
